### New Addition

In [0]:
# Configures the reference Silver pipeline and defines all Bronze, Silver, and quarantine table names.

from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.conf.set("spark.sql.session.timeZone", "UTC")

CATALOG = "training_lab_classic"
SCHEMA = "cold_chain"

NS = f"{CATALOG}.{SCHEMA}"

BRONZE_PRODUCTS_TABLE = f"{NS}.bronze_product_limits"
BRONZE_SHIPMENTS_TABLE = f"{NS}.bronze_shipments"
BRONZE_DELIVERIES_TABLE = f"{NS}.bronze_delivery_events"

SILVER_PRODUCTS_TABLE = f"{NS}.silver_product_limits"
SILVER_SHIPMENTS_TABLE = f"{NS}.silver_shipments"
SILVER_DELIVERIES_TABLE = f"{NS}.silver_delivery_events"
SILVER_SENSOR_TABLE = f"{NS}.silver_sensor_events"
SILVER_SENSOR_ENRICHED_TABLE = f"{NS}.silver_sensor_enriched"

QUARANTINE_PRODUCTS_TABLE = f"{NS}.quarantine_product_limits"
QUARANTINE_SHIPMENTS_TABLE = f"{NS}.quarantine_shipments"
QUARANTINE_DELIVERIES_TABLE = f"{NS}.quarantine_delivery_events"
QUARANTINE_SENSOR_TABLE = f"{NS}.quarantine_sensor_events"

print("Namespace:", NS)

In [0]:
# Verifies all required upstream tables and loads the Bronze reference datasets.

required_input_tables = [
    BRONZE_PRODUCTS_TABLE,
    BRONZE_SHIPMENTS_TABLE,
    BRONZE_DELIVERIES_TABLE,
    SILVER_SENSOR_TABLE
]

missing_input_tables = [
    table_name
    for table_name in required_input_tables
    if not spark.catalog.tableExists(table_name)
]

assert len(missing_input_tables) == 0, (
    "Missing required input tables: "
    + ", ".join(missing_input_tables)
)

bronze_products = spark.table(
    BRONZE_PRODUCTS_TABLE
)

bronze_shipments = spark.table(
    BRONZE_SHIPMENTS_TABLE
)

bronze_deliveries = spark.table(
    BRONZE_DELIVERIES_TABLE
)

silver_sensor_events = spark.table(
    SILVER_SENSOR_TABLE
)

print("Bronze products   :", bronze_products.count())
print("Bronze shipments  :", bronze_shipments.count())
print("Bronze deliveries :", bronze_deliveries.count())
print("Silver sensors    :", silver_sensor_events.count())

In [0]:
# Validates product identifiers and temperature ranges using generic data-quality rules.

validated_products = (
    bronze_products
    .withColumn(
        "product_id_clean",
        F.when(
            F.col("product_id").isNull(),
            F.lit(None).cast("string")
        ).otherwise(
            F.trim(F.col("product_id"))
        )
    )
    .withColumn(
        "rejection_reason",
        F.when(
            F.col("product_id_clean").isNull()
            | (F.col("product_id_clean") == ""),
            F.lit("MISSING_PRODUCT_ID")
        )
        .when(
            F.col("min_temperature").isNull(),
            F.lit("MISSING_MIN_TEMPERATURE")
        )
        .when(
            F.col("max_temperature").isNull(),
            F.lit("MISSING_MAX_TEMPERATURE")
        )
        .when(
            F.col("min_temperature")
            > F.col("max_temperature"),
            F.lit("INVALID_TEMPERATURE_RANGE")
        )
        .otherwise(
            F.lit(None).cast("string")
        )
    )
)

product_validation_summary = (
    validated_products
    .withColumn(
        "validation_status",
        F.when(
            F.col("rejection_reason").isNull(),
            F.lit("VALID")
        ).otherwise(
            F.lit("REJECTED")
        )
    )
    .groupBy(
        "validation_status",
        "rejection_reason"
    )
    .count()
    .orderBy(
        "validation_status",
        "rejection_reason"
    )
)

display(product_validation_summary)

In [0]:
# Splits validated products into clean Silver records and auditable quarantine records.

valid_products = (
    validated_products
    .filter(
        F.col("rejection_reason").isNull()
    )
)

quarantine_products = (
    validated_products
    .filter(
        F.col("rejection_reason").isNotNull()
    )
)

silver_product_limits = (
    valid_products
    .select(
        F.col("product_id_clean").alias("product_id"),
        F.col("min_temperature"),
        F.col("max_temperature"),
        F.col("_source_file"),
        F.col("_processing_time"),
        F.col("_run_id")
    )
)

quarantine_product_limits = (
    quarantine_products
    .select(
        F.col("product_id"),
        F.col("min_temperature"),
        F.col("max_temperature"),
        F.col("rejection_reason"),
        F.col("_source_file"),
        F.col("_processing_time"),
        F.col("_run_id")
    )
)

bronze_product_count = bronze_products.count()
silver_product_count = silver_product_limits.count()
quarantine_product_count = quarantine_product_limits.count()

assert (
    silver_product_count
    + quarantine_product_count
    == bronze_product_count
), (
    f"Product reconciliation failed: "
    f"Silver={silver_product_count}, "
    f"Quarantine={quarantine_product_count}, "
    f"Bronze={bronze_product_count}"
)

print("Bronze products     :", bronze_product_count)
print("Silver products     :", silver_product_count)
print("Quarantine products :", quarantine_product_count)
print("Product reconciliation passed.")

In [0]:
# Persists the validated product Silver and product quarantine datasets as Delta tables.

(
    silver_product_limits.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        SILVER_PRODUCTS_TABLE
    )
)

(
    quarantine_product_limits.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        QUARANTINE_PRODUCTS_TABLE
    )
)

print("Written:", SILVER_PRODUCTS_TABLE)
print("Written:", QUARANTINE_PRODUCTS_TABLE)

In [0]:
# Validates shipments against required fields, timestamps, and the validated Silver product reference.

silver_products = spark.table(
    SILVER_PRODUCTS_TABLE
)

valid_product_ids = (
    silver_products
    .select(
        F.col("product_id")
        .alias("valid_product_id")
    )
    .dropDuplicates(
        ["valid_product_id"]
    )
)

shipments_checked = (
    bronze_shipments.alias("shipment")
    .join(
        valid_product_ids.alias("product"),
        F.trim(F.col("shipment.product_id"))
        == F.col("product.valid_product_id"),
        "left"
    )
    .select(
        F.col("shipment.*"),
        F.col("product.valid_product_id")
    )
)

validated_shipments = (
    shipments_checked
    .withColumn(
        "shipment_id_clean",
        F.when(
            F.col("shipment_id").isNull(),
            F.lit(None).cast("string")
        ).otherwise(
            F.trim(F.col("shipment_id"))
        )
    )
    .withColumn(
        "product_id_clean",
        F.when(
            F.col("product_id").isNull(),
            F.lit(None).cast("string")
        ).otherwise(
            F.trim(F.col("product_id"))
        )
    )
    .withColumn(
        "rejection_reason",
        F.when(
            F.col("shipment_id_clean").isNull()
            | (F.col("shipment_id_clean") == ""),
            F.lit("MISSING_SHIPMENT_ID")
        )
        .when(
            F.col("product_id_clean").isNull()
            | (F.col("product_id_clean") == ""),
            F.lit("MISSING_PRODUCT_ID")
        )
        .when(
            F.col("valid_product_id").isNull(),
            F.lit("UNKNOWN_PRODUCT")
        )
        .when(
            F.col("dispatch_time").isNull(),
            F.lit("INVALID_DISPATCH_TIME")
        )
        .when(
            F.col("expected_arrival").isNull(),
            F.lit("INVALID_EXPECTED_ARRIVAL")
        )
        .when(
            F.col("expected_arrival")
            < F.col("dispatch_time"),
            F.lit("INVALID_SHIPMENT_TIME")
        )
        .otherwise(
            F.lit(None).cast("string")
        )
    )
)

shipment_validation_summary = (
    validated_shipments
    .withColumn(
        "validation_status",
        F.when(
            F.col("rejection_reason").isNull(),
            F.lit("VALID")
        ).otherwise(
            F.lit("REJECTED")
        )
    )
    .groupBy(
        "validation_status",
        "rejection_reason"
    )
    .count()
    .orderBy(
        "validation_status",
        "rejection_reason"
    )
)

display(shipment_validation_summary)

In [0]:
# Splits validated shipments into clean Silver records and shipment quarantine records.

valid_shipments = (
    validated_shipments
    .filter(
        F.col("rejection_reason").isNull()
    )
)

quarantine_shipments = (
    validated_shipments
    .filter(
        F.col("rejection_reason").isNotNull()
    )
)

silver_shipments = (
    valid_shipments
    .select(
        F.col("shipment_id_clean").alias("shipment_id"),
        F.col("product_id_clean").alias("product_id"),
        F.col("vehicle_id"),
        F.col("origin"),
        F.col("destination"),
        F.col("dispatch_time"),
        F.col("expected_arrival"),
        F.col("_source_file"),
        F.col("_processing_time"),
        F.col("_run_id")
    )
)

quarantine_shipments_final = (
    quarantine_shipments
    .select(
        F.col("shipment_id"),
        F.col("product_id"),
        F.col("vehicle_id"),
        F.col("origin"),
        F.col("destination"),
        F.col("dispatch_time"),
        F.col("expected_arrival"),
        F.col("rejection_reason"),
        F.col("_source_file"),
        F.col("_processing_time"),
        F.col("_run_id")
    )
)

bronze_shipment_count = bronze_shipments.count()
silver_shipment_count = silver_shipments.count()
quarantine_shipment_count = quarantine_shipments_final.count()

assert (
    silver_shipment_count
    + quarantine_shipment_count
    == bronze_shipment_count
), (
    f"Shipment reconciliation failed: "
    f"Silver={silver_shipment_count}, "
    f"Quarantine={quarantine_shipment_count}, "
    f"Bronze={bronze_shipment_count}"
)

print("Bronze shipments     :", bronze_shipment_count)
print("Silver shipments     :", silver_shipment_count)
print("Quarantine shipments :", quarantine_shipment_count)
print("Shipment reconciliation passed.")

In [0]:
# Persists clean shipment records and rejected shipment records as Delta tables.

(
    silver_shipments.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        SILVER_SHIPMENTS_TABLE
    )
)

(
    quarantine_shipments_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        QUARANTINE_SHIPMENTS_TABLE
    )
)

print("Written:", SILVER_SHIPMENTS_TABLE)
print("Written:", QUARANTINE_SHIPMENTS_TABLE)

In [0]:
# Validates delivery events against required fields, allowed status, and validated Silver shipments.

saved_silver_shipments = spark.table(
    SILVER_SHIPMENTS_TABLE
)

valid_shipment_ids = (
    saved_silver_shipments
    .select(
        F.col("shipment_id")
        .alias("valid_shipment_id")
    )
    .dropDuplicates(
        ["valid_shipment_id"]
    )
)

deliveries_checked = (
    bronze_deliveries.alias("delivery")
    .join(
        valid_shipment_ids.alias("shipment"),
        F.trim(F.col("delivery.shipment_id"))
        == F.col("shipment.valid_shipment_id"),
        "left"
    )
    .select(
        F.col("delivery.*"),
        F.col("shipment.valid_shipment_id")
    )
)

validated_deliveries = (
    deliveries_checked
    .withColumn(
        "event_id_clean",
        F.when(
            F.col("event_id").isNull(),
            F.lit(None).cast("string")
        ).otherwise(
            F.trim(F.col("event_id"))
        )
    )
    .withColumn(
        "shipment_id_clean",
        F.when(
            F.col("shipment_id").isNull(),
            F.lit(None).cast("string")
        ).otherwise(
            F.trim(F.col("shipment_id"))
        )
    )
    .withColumn(
        "status_clean",
        F.when(
            F.col("status").isNull(),
            F.lit(None).cast("string")
        ).otherwise(
            F.upper(
                F.trim(
                    F.col("status")
                )
            )
        )
    )
    .withColumn(
        "rejection_reason",
        F.when(
            F.col("event_id_clean").isNull()
            | (F.col("event_id_clean") == ""),
            F.lit("MISSING_DELIVERY_EVENT_ID")
        )
        .when(
            F.col("shipment_id_clean").isNull()
            | (F.col("shipment_id_clean") == ""),
            F.lit("MISSING_SHIPMENT_ID")
        )
        .when(
            F.col("valid_shipment_id").isNull(),
            F.lit("UNKNOWN_SHIPMENT")
        )
        .when(
            F.col("event_time").isNull(),
            F.lit("INVALID_EVENT_TIME")
        )
        .when(
            F.col("status_clean").isNull()
            | (F.col("status_clean") == ""),
            F.lit("MISSING_DELIVERY_STATUS")
        )
        .when(
            F.col("status_clean") != "DELIVERED",
            F.lit("INVALID_DELIVERY_STATUS")
        )
        .otherwise(
            F.lit(None).cast("string")
        )
    )
)

delivery_validation_summary = (
    validated_deliveries
    .withColumn(
        "validation_status",
        F.when(
            F.col("rejection_reason").isNull(),
            F.lit("VALID")
        ).otherwise(
            F.lit("REJECTED")
        )
    )
    .groupBy(
        "validation_status",
        "rejection_reason"
    )
    .count()
    .orderBy(
        "validation_status",
        "rejection_reason"
    )
)

display(delivery_validation_summary)

In [0]:
# Splits validated deliveries into clean Silver events and delivery quarantine records.

valid_deliveries = (
    validated_deliveries
    .filter(
        F.col("rejection_reason").isNull()
    )
)

quarantine_deliveries = (
    validated_deliveries
    .filter(
        F.col("rejection_reason").isNotNull()
    )
)

silver_delivery_events = (
    valid_deliveries
    .select(
        F.col("event_id_clean").alias("event_id"),
        F.col("shipment_id_clean").alias("shipment_id"),
        F.col("status_clean").alias("status"),
        F.col("event_time"),
        F.col("_source_file"),
        F.col("_processing_time"),
        F.col("_run_id")
    )
)

quarantine_delivery_events = (
    quarantine_deliveries
    .select(
        F.col("event_id"),
        F.col("shipment_id"),
        F.col("status"),
        F.col("event_time"),
        F.col("rejection_reason"),
        F.col("_source_file"),
        F.col("_processing_time"),
        F.col("_run_id")
    )
)

bronze_delivery_count = bronze_deliveries.count()
silver_delivery_count = silver_delivery_events.count()
quarantine_delivery_count = quarantine_delivery_events.count()

assert (
    silver_delivery_count
    + quarantine_delivery_count
    == bronze_delivery_count
), (
    f"Delivery reconciliation failed: "
    f"Silver={silver_delivery_count}, "
    f"Quarantine={quarantine_delivery_count}, "
    f"Bronze={bronze_delivery_count}"
)

print("Bronze deliveries     :", bronze_delivery_count)
print("Silver deliveries     :", silver_delivery_count)
print("Quarantine deliveries :", quarantine_delivery_count)
print("Delivery reconciliation passed.")

In [0]:
# Persists clean delivery events and rejected delivery records as Delta tables.

(
    silver_delivery_events.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        SILVER_DELIVERIES_TABLE
    )
)

(
    quarantine_delivery_events.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        QUARANTINE_DELIVERIES_TABLE
    )
)

print("Written:", SILVER_DELIVERIES_TABLE)
print("Written:", QUARANTINE_DELIVERIES_TABLE)

In [0]:
# Enriches validated sensor events with shipment information
# and product temperature limits.

sensors = spark.table(
    SILVER_SENSOR_TABLE
)

shipments = spark.table(
    SILVER_SHIPMENTS_TABLE
)

products = spark.table(
    SILVER_PRODUCTS_TABLE
)

# Confirms that incremental metadata reached Silver.
required_incremental_columns = {
    "batch_id",
    "load_mode"
}

missing_incremental_columns = (
    required_incremental_columns
    - set(sensors.columns)
)

assert not missing_incremental_columns, (
    "Silver sensor table is missing columns: "
    + ", ".join(
        sorted(missing_incremental_columns)
    )
)

shipment_lookup = (
    shipments
    .select(
        "shipment_id",
        "product_id",
        "vehicle_id",
        "origin",
        "destination",
        "dispatch_time",
        "expected_arrival"
    )
    .dropDuplicates(
        ["shipment_id"]
    )
)

product_lookup = (
    products
    .select(
        "product_id",
        "min_temperature",
        "max_temperature"
    )
    .dropDuplicates(
        ["product_id"]
    )
)

sensor_with_shipment = (
    sensors.alias("sensor")
    .join(
        shipment_lookup.alias("shipment"),
        F.col("sensor.shipment_id")
        == F.col("shipment.shipment_id"),
        "inner"
    )
    .select(
        F.col(
            "sensor.event_id"
        ).alias(
            "event_id"
        ),
        F.col(
            "sensor.shipment_id"
        ).alias(
            "shipment_id"
        ),
        F.col(
            "sensor.sensor_id"
        ).alias(
            "sensor_id"
        ),
        F.col(
            "sensor.event_time"
        ).alias(
            "event_time"
        ),
        F.col(
            "sensor.temperature_c"
        ).alias(
            "temperature_c"
        ),

        # Incremental metadata preserved from Silver
        F.col(
            "sensor.batch_id"
        ).alias(
            "batch_id"
        ),
        F.col(
            "sensor.load_mode"
        ).alias(
            "load_mode"
        ),

        F.col(
            "shipment.product_id"
        ).alias(
            "product_id"
        ),
        F.col(
            "shipment.vehicle_id"
        ).alias(
            "vehicle_id"
        ),
        F.col(
            "shipment.origin"
        ).alias(
            "origin"
        ),
        F.col(
            "shipment.destination"
        ).alias(
            "destination"
        ),
        F.col(
            "shipment.dispatch_time"
        ).alias(
            "dispatch_time"
        ),
        F.col(
            "shipment.expected_arrival"
        ).alias(
            "expected_arrival"
        ),
        F.col(
            "sensor._source_file"
        ).alias(
            "_source_file"
        ),
        F.col(
            "sensor._processing_time"
        ).alias(
            "_processing_time"
        ),
        F.col(
            "sensor._run_id"
        ).alias(
            "_run_id"
        )
    )
)

# Verify before performing the product join.
assert "batch_id" in sensor_with_shipment.columns
assert "load_mode" in sensor_with_shipment.columns

sensor_enriched = (
    sensor_with_shipment.alias("sensor")
    .join(
        product_lookup.alias("product"),
        F.col("sensor.product_id")
        == F.col("product.product_id"),
        "inner"
    )
    .select(
        F.col(
            "sensor.event_id"
        ).alias(
            "event_id"
        ),
        F.col(
            "sensor.shipment_id"
        ).alias(
            "shipment_id"
        ),
        F.col(
            "sensor.sensor_id"
        ).alias(
            "sensor_id"
        ),
        F.col(
            "sensor.event_time"
        ).alias(
            "event_time"
        ),
        F.col(
            "sensor.temperature_c"
        ).alias(
            "temperature_c"
        ),
        F.col(
            "sensor.batch_id"
        ).alias(
            "batch_id"
        ),
        F.col(
            "sensor.load_mode"
        ).alias(
            "load_mode"
        ),
        F.col(
            "sensor.product_id"
        ).alias(
            "product_id"
        ),
        F.col(
            "product.min_temperature"
        ).alias(
            "min_temperature"
        ),
        F.col(
            "product.max_temperature"
        ).alias(
            "max_temperature"
        ),
        F.col(
            "sensor.vehicle_id"
        ).alias(
            "vehicle_id"
        ),
        F.col(
            "sensor.origin"
        ).alias(
            "origin"
        ),
        F.col(
            "sensor.destination"
        ).alias(
            "destination"
        ),
        F.col(
            "sensor.dispatch_time"
        ).alias(
            "dispatch_time"
        ),
        F.col(
            "sensor.expected_arrival"
        ).alias(
            "expected_arrival"
        ),
        F.col(
            "sensor._source_file"
        ).alias(
            "_source_file"
        ),
        F.col(
            "sensor._processing_time"
        ).alias(
            "_processing_time"
        ),
        F.col(
            "sensor._run_id"
        ).alias(
            "_run_id"
        )
    )
    .withColumn(
        "is_temperature_exception",
        (
            F.col("temperature_c")
            < F.col("min_temperature")
        )
        |
        (
            F.col("temperature_c")
            > F.col("max_temperature")
        )
    )
)

sensor_count = sensors.count()

shipment_enriched_count = (
    sensor_with_shipment.count()
)

fully_enriched_count = (
    sensor_enriched.count()
)

assert shipment_enriched_count == sensor_count, (
    f"Sensor-to-shipment enrichment lost records: "
    f"Sensor={sensor_count}, "
    f"AfterShipmentJoin={shipment_enriched_count}"
)

assert fully_enriched_count == sensor_count, (
    f"Sensor-to-product enrichment lost records: "
    f"Sensor={sensor_count}, "
    f"AfterProductJoin={fully_enriched_count}"
)

print

In [0]:
# Persists the fully enriched sensor dataset with generic temperature-exception classification.

(
    sensor_enriched.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        SILVER_SENSOR_ENRICHED_TABLE
    )
)

saved_enriched = spark.table(
    SILVER_SENSOR_ENRICHED_TABLE
)

expected_enriched_count = (
    spark.table(
        SILVER_SENSOR_TABLE
    ).count()
)

saved_enriched_count = (
    saved_enriched.count()
)

assert saved_enriched_count == expected_enriched_count, (
    f"Persisted enrichment count mismatch: "
    f"Sensor={expected_enriched_count}, "
    f"Enriched={saved_enriched_count}"
)

print(
    "Enriched sensor rows:",
    saved_enriched_count
)

print(
    "silver_sensor_enriched written successfully."
)

display(
    saved_enriched
    .select(
        "event_id",
        "shipment_id",
        "product_id",
        "temperature_c",
        "min_temperature",
        "max_temperature",
        "is_temperature_exception"
    )
    .orderBy(
        "event_time",
        "event_id"
    )
)

In [0]:
# Validates temperature-exception classification and reference enrichment without relying on fixture-specific event IDs.

enriched = spark.table(
    SILVER_SENSOR_ENRICHED_TABLE
)

missing_reference_data_count = (
    enriched
    .filter(
        F.col("product_id").isNull()
        | F.col("min_temperature").isNull()
        | F.col("max_temperature").isNull()
        | F.col("shipment_id").isNull()
    )
    .count()
)

incorrect_exception_true_count = (
    enriched
    .filter(
        (
            F.col("is_temperature_exception") == True
        )
        &
        (
            F.col("temperature_c")
            >= F.col("min_temperature")
        )
        &
        (
            F.col("temperature_c")
            <= F.col("max_temperature")
        )
    )
    .count()
)

incorrect_exception_false_count = (
    enriched
    .filter(
        (
            F.col("is_temperature_exception") == False
        )
        &
        (
            (
                F.col("temperature_c")
                < F.col("min_temperature")
            )
            |
            (
                F.col("temperature_c")
                > F.col("max_temperature")
            )
        )
    )
    .count()
)

temperature_exception_count = (
    enriched
    .filter(
        F.col("is_temperature_exception") == True
    )
    .count()
)

normal_temperature_count = (
    enriched
    .filter(
        F.col("is_temperature_exception") == False
    )
    .count()
)

assert missing_reference_data_count == 0, (
    f"Enriched Silver contains "
    f"{missing_reference_data_count} rows "
    f"with missing reference data"
)

assert incorrect_exception_true_count == 0, (
    f"Found {incorrect_exception_true_count} rows "
    f"incorrectly marked as temperature exceptions"
)

assert incorrect_exception_false_count == 0, (
    f"Found {incorrect_exception_false_count} "
    f"out-of-range rows not marked as exceptions"
)

assert (
    temperature_exception_count
    + normal_temperature_count
    == enriched.count()
), (
    "Temperature classification reconciliation failed"
)

print("==============================================")
print("ENRICHED SENSOR QUALITY CHECK")
print("==============================================")

print(
    "Enriched sensor rows             :",
    enriched.count()
)

print(
    "Temperature exceptions           :",
    temperature_exception_count
)

print(
    "Normal temperature readings      :",
    normal_temperature_count
)

print(
    "Missing reference data           :",
    missing_reference_data_count
)

print(
    "False positive exceptions        :",
    incorrect_exception_true_count
)

print(
    "Missed temperature exceptions    :",
    incorrect_exception_false_count
)

print()
print("Enriched sensor validation passed.")

In [0]:
# Identifies validated shipments with and without corresponding delivery events using generic joins.

silver_shipments_check = spark.table(
    SILVER_SHIPMENTS_TABLE
)

silver_deliveries_check = spark.table(
    SILVER_DELIVERIES_TABLE
)

delivered_shipment_ids = (
    silver_deliveries_check
    .select(
        "shipment_id"
    )
    .dropDuplicates(
        ["shipment_id"]
    )
)

undelivered_shipments = (
    silver_shipments_check.alias("shipment")
    .join(
        delivered_shipment_ids.alias("delivery"),
        F.col("shipment.shipment_id")
        == F.col("delivery.shipment_id"),
        "left_anti"
    )
    .select(
        "shipment.*"
    )
)

delivered_shipment_count = (
    delivered_shipment_ids.count()
)

undelivered_shipment_count = (
    undelivered_shipments.count()
)

print("==============================================")
print("DELIVERY COVERAGE")
print("==============================================")

print(
    "Total valid shipments     :",
    silver_shipments_check.count()
)

print(
    "Shipments with delivery   :",
    delivered_shipment_count
)

print(
    "Shipments without delivery:",
    undelivered_shipment_count
)

display(
    undelivered_shipments
    .select(
        "shipment_id",
        "product_id",
        "origin",
        "destination",
        "dispatch_time",
        "expected_arrival"
    )
    .orderBy(
        "expected_arrival"
    )
)

In [0]:
# Produces the final scale-independent Reference Silver certificate across products, shipments, deliveries, sensors, and enrichment.

bronze_products_final = spark.table(
    BRONZE_PRODUCTS_TABLE
)

bronze_shipments_final = spark.table(
    BRONZE_SHIPMENTS_TABLE
)

bronze_deliveries_final = spark.table(
    BRONZE_DELIVERIES_TABLE
)

silver_products_final = spark.table(
    SILVER_PRODUCTS_TABLE
)

silver_shipments_final = spark.table(
    SILVER_SHIPMENTS_TABLE
)

silver_deliveries_final = spark.table(
    SILVER_DELIVERIES_TABLE
)

silver_sensors_final = spark.table(
    SILVER_SENSOR_TABLE
)

silver_enriched_final = spark.table(
    SILVER_SENSOR_ENRICHED_TABLE
)

quarantine_products_final = spark.table(
    QUARANTINE_PRODUCTS_TABLE
)

quarantine_shipments_final = spark.table(
    QUARANTINE_SHIPMENTS_TABLE
)

quarantine_deliveries_final = spark.table(
    QUARANTINE_DELIVERIES_TABLE
)

bronze_product_count = bronze_products_final.count()
bronze_shipment_count = bronze_shipments_final.count()
bronze_delivery_count = bronze_deliveries_final.count()

silver_product_count = silver_products_final.count()
silver_shipment_count = silver_shipments_final.count()
silver_delivery_count = silver_deliveries_final.count()
silver_sensor_count = silver_sensors_final.count()
silver_enriched_count = silver_enriched_final.count()

quarantine_product_count = quarantine_products_final.count()
quarantine_shipment_count = quarantine_shipments_final.count()
quarantine_delivery_count = quarantine_deliveries_final.count()

product_reconciled_count = (
    silver_product_count
    + quarantine_product_count
)

shipment_reconciled_count = (
    silver_shipment_count
    + quarantine_shipment_count
)

delivery_reconciled_count = (
    silver_delivery_count
    + quarantine_delivery_count
)

missing_product_reference_count = (
    silver_shipments_final.alias("shipment")
    .join(
        silver_products_final
        .select("product_id")
        .dropDuplicates(["product_id"])
        .alias("product"),
        F.col("shipment.product_id")
        == F.col("product.product_id"),
        "left_anti"
    )
    .count()
)

missing_delivery_shipment_reference_count = (
    silver_deliveries_final.alias("delivery")
    .join(
        silver_shipments_final
        .select("shipment_id")
        .dropDuplicates(["shipment_id"])
        .alias("shipment"),
        F.col("delivery.shipment_id")
        == F.col("shipment.shipment_id"),
        "left_anti"
    )
    .count()
)

missing_enrichment_reference_count = (
    silver_enriched_final
    .filter(
        F.col("shipment_id").isNull()
        | F.col("product_id").isNull()
        | F.col("min_temperature").isNull()
        | F.col("max_temperature").isNull()
    )
    .count()
)

duplicate_product_ids = (
    silver_products_final
    .groupBy("product_id")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

duplicate_shipment_ids = (
    silver_shipments_final
    .groupBy("shipment_id")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

duplicate_sensor_event_ids = (
    silver_sensors_final
    .groupBy("event_id")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

incorrect_temperature_flags = (
    silver_enriched_final
    .filter(
        (
            (
                F.col("temperature_c")
                < F.col("min_temperature")
            )
            |
            (
                F.col("temperature_c")
                > F.col("max_temperature")
            )
        )
        != F.col("is_temperature_exception")
    )
    .count()
)

temperature_exception_count = (
    silver_enriched_final
    .filter(
        F.col("is_temperature_exception") == True
    )
    .count()
)

assert product_reconciled_count == bronze_product_count, (
    f"Product reconciliation failed: "
    f"Silver={silver_product_count}, "
    f"Quarantine={quarantine_product_count}, "
    f"Bronze={bronze_product_count}"
)

assert shipment_reconciled_count == bronze_shipment_count, (
    f"Shipment reconciliation failed: "
    f"Silver={silver_shipment_count}, "
    f"Quarantine={quarantine_shipment_count}, "
    f"Bronze={bronze_shipment_count}"
)

assert delivery_reconciled_count == bronze_delivery_count, (
    f"Delivery reconciliation failed: "
    f"Silver={silver_delivery_count}, "
    f"Quarantine={quarantine_delivery_count}, "
    f"Bronze={bronze_delivery_count}"
)

assert silver_enriched_count == silver_sensor_count, (
    f"Sensor enrichment lost records: "
    f"Sensor={silver_sensor_count}, "
    f"Enriched={silver_enriched_count}"
)

assert missing_product_reference_count == 0, (
    f"Silver shipments contain "
    f"{missing_product_reference_count} "
    f"unknown product references"
)

assert missing_delivery_shipment_reference_count == 0, (
    f"Silver deliveries contain "
    f"{missing_delivery_shipment_reference_count} "
    f"unknown shipment references"
)

assert missing_enrichment_reference_count == 0, (
    f"Enriched sensors contain "
    f"{missing_enrichment_reference_count} "
    f"missing reference values"
)

assert duplicate_product_ids == 0, (
    f"Silver products contain "
    f"{duplicate_product_ids} duplicated product IDs"
)

assert duplicate_shipment_ids == 0, (
    f"Silver shipments contain "
    f"{duplicate_shipment_ids} duplicated shipment IDs"
)

assert duplicate_sensor_event_ids == 0, (
    f"Silver sensors contain "
    f"{duplicate_sensor_event_ids} duplicated event IDs"
)

assert incorrect_temperature_flags == 0, (
    f"Found {incorrect_temperature_flags} "
    f"incorrect temperature classifications"
)

print("================================================")
print("FINAL REFERENCE SILVER CERTIFICATE")
print("================================================")

print()
print("PRODUCT PIPELINE")
print("------------------------------------------------")
print("Bronze products                 :", bronze_product_count)
print("Silver products                 :", silver_product_count)
print("Quarantined products            :", quarantine_product_count)
print("Duplicate product IDs           :", duplicate_product_ids)

print()
print("SHIPMENT PIPELINE")
print("------------------------------------------------")
print("Bronze shipments                :", bronze_shipment_count)
print("Silver shipments                :", silver_shipment_count)
print("Quarantined shipments           :", quarantine_shipment_count)
print("Duplicate shipment IDs          :", duplicate_shipment_ids)

print()
print("DELIVERY PIPELINE")
print("------------------------------------------------")
print("Bronze deliveries               :", bronze_delivery_count)
print("Silver deliveries               :", silver_delivery_count)
print("Quarantined deliveries          :", quarantine_delivery_count)

print()
print("SENSOR PIPELINE")
print("------------------------------------------------")
print("Silver sensors                  :", silver_sensor_count)
print("Enriched sensors                :", silver_enriched_count)
print("Duplicate sensor event IDs      :", duplicate_sensor_event_ids)
print("Temperature exceptions          :", temperature_exception_count)

print()
print("REFERENTIAL INTEGRITY")
print("------------------------------------------------")
print("Unknown product references      :", missing_product_reference_count)
print("Unknown shipment deliveries     :", missing_delivery_shipment_reference_count)
print("Missing enrichment references   :", missing_enrichment_reference_count)

print()
print("BUSINESS-RULE VALIDATION")
print("------------------------------------------------")
print("Incorrect temperature flags     :", incorrect_temperature_flags)

print()
print("================================================")
print("REFERENCE SILVER VALIDATION PASSED")
print("================================================")

### New till here

In [0]:
from pyspark.sql import functions as F


# ---------------------------------------------------------
# Table configuration
# ---------------------------------------------------------

BRONZE_TABLE = (
    "training_lab_classic."
    "cold_chain."
    "bronze_sensor_events"
)

SILVER_TABLE = (
    "training_lab_classic."
    "cold_chain."
    "silver_sensor_events"
)

QUARANTINE_TABLE = (
    "training_lab_classic."
    "cold_chain."
    "quarantine_sensor_events"
)

ENRICHED_TABLE = (
    "training_lab_classic."
    "cold_chain."
    "silver_sensor_enriched"
)

HISTORICAL_BATCH_ID = "historical_001"


# ---------------------------------------------------------
# Load current Delta tables
# ---------------------------------------------------------

bronze = spark.table(
    BRONZE_TABLE
)

silver = spark.table(
    SILVER_TABLE
)

quarantine = spark.table(
    QUARANTINE_TABLE
)

enriched = spark.table(
    ENRICHED_TABLE
)


# ---------------------------------------------------------
# Calculate historical batch counts
# ---------------------------------------------------------

bronze_historical = (
    bronze
    .filter(
        F.col("batch_id")
        == HISTORICAL_BATCH_ID
    )
    .count()
)

silver_historical = (
    silver
    .filter(
        F.col("batch_id")
        == HISTORICAL_BATCH_ID
    )
    .count()
)

quarantine_historical = (
    quarantine
    .filter(
        F.col("batch_id")
        == HISTORICAL_BATCH_ID
    )
    .count()
)

enriched_historical = (
    enriched
    .filter(
        F.col("batch_id")
        == HISTORICAL_BATCH_ID
    )
    .count()
)

reconciled_historical = (
    silver_historical
    + quarantine_historical
)


# ---------------------------------------------------------
# Print validation counts
# ---------------------------------------------------------

print("==============================================")
print("HISTORICAL ENRICHMENT VALIDATION")
print("==============================================")

print(
    "Historical batch ID :",
    HISTORICAL_BATCH_ID
)

print(
    "Bronze historical   :",
    bronze_historical
)

print(
    "Silver accepted     :",
    silver_historical
)

print(
    "Silver quarantine   :",
    quarantine_historical
)

print(
    "Silver reconciled   :",
    reconciled_historical
)

print(
    "Enriched rows       :",
    enriched_historical
)


# ---------------------------------------------------------
# Required metadata validation
# ---------------------------------------------------------

required_enriched_columns = {
    "batch_id",
    "load_mode",
}

missing_enriched_columns = (
    required_enriched_columns
    - set(enriched.columns)
)

assert not missing_enriched_columns, (
    "Required metadata columns are missing from "
    "the enriched table: "
    + ", ".join(
        sorted(
            missing_enriched_columns
        )
    )
)


# ---------------------------------------------------------
# Historical baseline validations
# ---------------------------------------------------------

assert bronze_historical == 10000, (
    "Historical Bronze count mismatch: "
    f"expected=10000, "
    f"actual={bronze_historical}"
)

assert reconciled_historical == bronze_historical, (
    "Silver reconciliation failed: "
    f"accepted={silver_historical}, "
    f"quarantined={quarantine_historical}, "
    f"reconciled={reconciled_historical}, "
    f"bronze={bronze_historical}"
)

assert enriched_historical == silver_historical, (
    "Enrichment lost or duplicated accepted rows: "
    f"enriched={enriched_historical}, "
    f"silver={silver_historical}"
)


# ---------------------------------------------------------
# Batch metadata validations
# ---------------------------------------------------------

invalid_enriched_batch_rows = (
    enriched
    .filter(
        F.col("batch_id")
        == HISTORICAL_BATCH_ID
    )
    .filter(
        F.col("load_mode") != "historical"
    )
    .count()
)

assert invalid_enriched_batch_rows == 0, (
    "Some historical enriched rows have an "
    "incorrect load_mode: "
    f"{invalid_enriched_batch_rows}"
)

null_batch_id_rows = (
    enriched
    .filter(
        F.col("batch_id").isNull()
    )
    .count()
)

null_load_mode_rows = (
    enriched
    .filter(
        F.col("load_mode").isNull()
    )
    .count()
)

assert null_batch_id_rows == 0, (
    "Enriched output contains rows with "
    "a null batch_id: "
    f"{null_batch_id_rows}"
)

assert null_load_mode_rows == 0, (
    "Enriched output contains rows with "
    "a null load_mode: "
    f"{null_load_mode_rows}"
)


# ---------------------------------------------------------
# Final result
# ---------------------------------------------------------

print()
print("==============================================")
print("HISTORICAL ENRICHMENT VALIDATION: PASS")
print("==============================================")

print(
    "Accepted rows preserved:",
    enriched_historical
)

print(
    "Quarantine rows:",
    quarantine_historical
)

print(
    "Total reconciled:",
    reconciled_historical
)